# Lab 04 — The Normal Distribution

*Where the bell curve comes from, why it is everywhere, and how to generate one from uniform noise.*

## Objectives

1. Build a normal distribution by summing uniform random variables
2. See the Central Limit Theorem happen numerically
3. Compute and interpret mean, variance, and standard deviation
4. Fit a normal curve to a histogram

---

In [ ]:
# --- setup: run this cell first, every session ---------------------------------
import builtins as __builtin__
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import *

import torchvision
from torchvision import datasets, models, transforms
from torchvision.models import *

from scipy import stats, signal, integrate
from sympy import Symbol

from skimage import io
from skimage import io as skio
from skimage.transform import resize
from skimage.util import montage
from imageio.v2 import imread

from IPython.display import display, HTML

# One device for the whole notebook. A GPU runtime is faster, but nothing here
# requires one -- switch with Runtime -> Change runtime type -> T4 GPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def GPU(data):
    """A learnable float tensor on the active device."""
    return torch.tensor(data, requires_grad=True, dtype=torch.float, device=device)


def GPU_data(data):
    """A fixed (non-learnable) float tensor on the active device."""
    return torch.tensor(data, requires_grad=False, dtype=torch.float, device=device)


print('Running on:', device)

# Normal Distribution

In [ ]:
# Define the parameters for the normal distribution
mean = 0  # Mean (mu) of the distribution
std_dev = 0.1  # Standard deviation (sigma) of the distribution

# Generate 1000 data points following a normal distribution with the specified mean and standard deviation
sample_size = 10000
data_points = np.random.normal(mean, std_dev, sample_size)

# Plot the histogram of the data points
bins_number = 30  # Number of bins for the histogram
hist_count, x, ignored = plt.hist(data_points, bins_number, density=True)


In [ ]:
x = np.linspace(-15,15,1000)

y = np.exp(-x**2)

In [ ]:
plt.plot(x,y)

In [ ]:
y2 = np.sin(10*x)

In [ ]:
plt.plot(x,y2)

In [ ]:
plt.plot(x,y*y2)

In [ ]:
y3 =  np.exp(- (x - 0) ** 2 )
plt.plot(x, y3, linewidth=2, color='r')

In [ ]:
# Define the parameters for the normal distribution
mean = 0  # Mean (mu) of the distribution
std_dev = 0.1  # Standard deviation (sigma) of the distribution

# Generate 1000 data points following a normal distribution with the specified mean and standard deviation
sample_size = 10000
data_points = np.random.normal(mean, std_dev, sample_size)

# Plot the histogram of the data points
bins_number = 30  # Number of bins for the histogram
hist_count, x, ignored = plt.hist(data_points, bins_number, density=True)

# Plot the probability density function of the normal distribution
normal_dist_curve = 1 / (std_dev * np.sqrt(2 * np.pi)) * np.exp(- (x - mean) ** 2 / (2 * std_dev ** 2))
plt.plot(x, normal_dist_curve, linewidth=2, color='r')

# Set the title and labels for the plot
plt.title('Normal Distribution Visualization')
plt.xlabel('Value')
plt.ylabel('Probability Density')

# Display the plot
plt.show()

In [ ]:
x = np.array([1,2,3,4])

In [ ]:
np.sum(x)

In [ ]:
x.shape[0]

In [ ]:
np.sum(x)/x.shape[0]

In [ ]:
def mean(x):
    return np.sum(x)/x.shape[0]

In [ ]:
mean(x)

In [ ]:
x

In [ ]:
x - mean(x)

In [ ]:
(x - mean(x))**2

In [ ]:
 def var(x):
    return mean((x - mean(x))**2)

In [ ]:
def std(x):
    return np.sqrt(var(x))

In [ ]:
mean(x)

In [ ]:
var(x)

In [ ]:
std(x)

### Uniform Random Numbers

In [ ]:
X = np.random.random(500000,)

In [ ]:
X

In [ ]:
plt.hist(X,100);

In [ ]:
np.random.random()

In [ ]:
r = np.random.random()

In [ ]:
r

In [ ]:
r = np.random.randint(1,10)

In [ ]:
r

In [ ]:
np.random.randn()

In [ ]:
numbers  = np.random.randn(2,4)

In [ ]:
numbers.shape

In [ ]:
numbers = numbers.reshape(-1)

In [ ]:
numbers.shape

In [ ]:
numbers= np.array([1,2,2,3,3,3,4,4,4,4])

In [ ]:
plt.hist(numbers,100);

In [ ]:
numbers = np.random.randn(100000,)
plt.hist(numbers,100);

In [ ]:
numbers = np.random.rand(1000000,)
plt.hist(numbers,100);

In [ ]:
np.random.seed(12345)
data = np.random.randn(2, 100)

plt.figure(1, figsize=(9, 9))

plt.subplot(2,2,1)
plt.hist(data[0])

plt.subplot(2,2,2)
plt.scatter(data[0], data[1])

plt.subplot(2,2,3)
plt.plot(data[0], data[1],'-')

plt.subplot(2,2,4)
plt.hist2d(data[0], data[1])

plt.show()

Normal Dist Data

In [ ]:
x = 10*np.random.randn(10000)

In [ ]:
plt.hist(x);

In [ ]:
x = np.random.rand(1000,)

In [ ]:
plt.hist(x);

In [ ]:
mu = 0  # mean of distribution
sigma = 1  # standard deviation of distribution
x = mu + sigma * np.random.randn(10000)

In [ ]:
n,bins,patches = plt.hist(x,bins=100)

In [ ]:
bins

In [ ]:
y = ((1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * (1 / sigma * (bins - mu))**2))
plt.plot(y)

In [ ]:
num_bins = 50
n,bins,patches = plt.hist(x, num_bins, density=1)
y = ((1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * (1 / sigma * (bins - mu))**2))
plt.plot(bins,y)

In [ ]:
x

In [ ]:
def mean(x):
    return np.sum(x)/x.shape[0]

In [ ]:
def var(x):
    return mean((x - mean(x))**2)

In [ ]:
def std(x):
    return np.sqrt(var(x))

In [ ]:
def median(x):
    n = len(x)
    sorted_x = np.sort(x)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_x[mid - 1] + sorted_x[mid]) / 2
    else:
        return sorted_x[mid]

In [ ]:
def mode(x):
    values, counts = np.unique(x, return_counts=True)
    max_count_index = np.argmax(counts)
    return values[max_count_index]

In [ ]:
def range1(x):
    return np.max(x) - np.min(x)

In [ ]:
data = np.array([1, 2, 2, 3, 4, 5, 5, 5, 6])

# Testing the functions
mean_value = mean(data)
var_value = var(data)
std_value = std(data)
median_value = median(data)
mode_value = mode(data)
range_value = range1(data)

mean_value, var_value, std_value, median_value, mode_value, range_value


---

## Homework

1. Sum 2, 5, and 50 uniforms. Plot all three histograms on one figure.
2. Collect 30 real measurements of something. Is the distribution normal? Argue with a plot.

Submit as a markdown report in your course GitHub repo, with plots committed alongside it.

<details><summary>Where this lab came from</summary>

- `_source/notebooks/Math24/Lab_2__1IEV3ROU.ipynb` cells 22–87 (60 kept)

</details>